# PV Project

## Model training

### Import Libraries & Load the data

In [ ]:
%pip install matplotlib seaborn

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

### Establish connection

In [4]:
# Establishing connection wit the database

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DB_PATH = PROJECT_ROOT / "data" / "faers_db.duckdb"

con = duckdb.connect(str(DB_PATH), read_only=True) # read_only=True for exploration, and read_only=False only when creating tables.

In [5]:
pd.set_option("display.max_columns", 60, "display.width", 220)
print(con.sql("SHOW TABLES").df())

                     name
0               case_base
1               case_drug
2           case_exposure
3              case_month
4            case_outcome
5                 case_pt
6             case_source
7              clean_demo
8                 demo_us
9             drug_detail
10               exposure
11             faers_flat
12                    m_N
13                   m_ev
14                  m_grp
15                 m_pair
16                 months
17            pair_counts
18              pair_grid
19             pair_month
20                ps_case
21          ps_case_event
22                ps_drug
23               raw_demo
24               raw_indi
25               raw_reac
26               raw_ther
27               ref_drug
28       ref_known_signal
29          signal_labels
30              signal_ts
31       study_case_event
32            study_cases
33        study_drug_rows
34   v_ingredient_summary
35   v_known_signal_check
36  v_known_signal_family
37        v_

### Validation before forcasting

In [8]:
# 1. ps_case_event reproduces pair_counts (expect 0)
print(con.sql("""SELECT COUNT(*) AS mismatched_pairs FROM pair_counts p
    LEFT JOIN v_pair_serious e USING (ingredient, pt) WHERE e.cases IS DISTINCT FROM p.a""").df())

# 2. Labels per ingredient
print(con.sql("SELECT ingredient, status, COUNT(*) n FROM signal_labels GROUP BY ALL ORDER BY 1, 2").df())

# 3. Known signals and lead time
print(con.sql("""SELECT ingredient, pt, a, round(ic025,2) ic025, signal, onset, status, lead_days
                 FROM v_known_signal_check ORDER BY ingredient, ic025 DESC""").df())

# 4. Drug profiles
print(con.sql("SELECT * FROM v_ingredient_summary ORDER BY ps_cases DESC").df())

   mismatched_pairs
0                 0
       ingredient        status    n
0   canagliflozin      emerging  245
1   canagliflozin         never  590
2   canagliflozin  pre-existing   79
3   dapagliflozin      emerging  142
4   dapagliflozin         never  362
5   dapagliflozin  pre-existing   45
6     dulaglutide      emerging  160
7     dulaglutide         never  666
8     dulaglutide  pre-existing    3
9   empagliflozin      emerging  178
10  empagliflozin         never  363
11  empagliflozin  pre-existing   12
12  ertugliflozin      emerging   23
13  ertugliflozin         never   26
14    liraglutide      emerging  127
15    liraglutide         never  540
16    liraglutide  pre-existing   35
17    semaglutide      emerging   89
18    semaglutide         never  185
       ingredient                                 pt     a  ic025  signal      onset        status  lead_days
0   canagliflozin                     TOE AMPUTATION  1915   7.55    True 2016-06-01      emerging        -43


In [7]:
ts = con.sql("SELECT * FROM signal_ts").df()
ts["m"] = pd.to_datetime(ts["m"])
ts = ts.sort_values(["ingredient", "pt", "m"]).reset_index(drop=True)

K = 6                                    # forecast horizon in months
key = ["ingredient", "pt"]
g = ts.groupby(key)

# features (all computed from data up to month m)
ts["new_cases"]   = g["a"].diff().fillna(ts["a"])
ts["new_3m"]      = g["new_cases"].transform(lambda s: s.rolling(3, min_periods=1).sum())
ts["ic025_d3"]    = g["IC025"].diff(3)
ts["sig_int"]     = ts["signal"].astype(int)

# target: signal at any of the next K months
ts["target"] = g["sig_int"].transform(
    lambda s: s[::-1].shift(1).rolling(K, min_periods=1).max()[::-1])

last = ts.m.max()
panel = ts[(ts.m >= "2015-01-01")                       # after burn-in
           & (ts.m <= last - pd.DateOffset(months=K))   # full horizon observed
           & (~ts.signal)                               # not already a signal at the origin
           & (ts.a >= 1)].copy()
print(len(panel), "origin-pair rows | positive rate:", panel.target.mean().round(3))
print(panel.groupby("ingredient").target.agg(["size", "mean"]).round(3))

131376 origin-pair rows | positive rate: 0.038
                size   mean
ingredient                 
canagliflozin  35037  0.032
dapagliflozin  20739  0.041
dulaglutide    27751  0.032
empagliflozin  17729  0.055
ertugliflozin    374  0.217
liraglutide    27499  0.029
semaglutide     2247  0.136


### Machine Learning

In [6]:

ts  = con.sql("SELECT * FROM signal_ts").df()
cls = con.sql("SELECT * FROM ref_drug").df()
ts["m"] = pd.to_datetime(ts["m"])
ts = ts.merge(cls, on="ingredient").sort_values(["ingredient","pt","m"]).reset_index(drop=True)

K, key = 6, ["ingredient","pt"]
g = ts.groupby(key)
ts["sig"]        = ts.signal.astype(int)
ts["new_cases"]  = g["a"].diff().fillna(ts["a"])
ts["new_3m"]     = g["new_cases"].transform(lambda s: s.rolling(3, min_periods=1).sum())
ts["ic025_d3"]   = g["IC025"].diff(3)
ts["ic025_d6"]   = g["IC025"].diff(6)
ts["age_m"]      = g.cumcount()                                  # months since the pair first appeared
ts["slope6"]     = ts["ic025_d6"] / 6
ts["trend_K"]    = ts["IC025"] + K * ts["slope6"].fillna(0)      # linear extrapolation K months ahead
ts["target"]     = g["sig"].transform(lambda s: s[::-1].shift(1).rolling(K, min_periods=1).max()[::-1])

# class effect: is the same event already a signal for another drug in the class?
cs = ts.groupby(["drug_class","pt","m"]).sig.sum().rename("cls_sig").reset_index()
ts = ts.merge(cs, on=["drug_class","pt","m"])
ts["other_cls_sig"] = ts.cls_sig - ts.sig

last = ts.m.max()
panel = ts[(ts.m >= "2015-01-01") & (ts.m <= last - pd.DateOffset(months=K))
           & (ts.sig == 0)].dropna(subset=["target"]).copy()
print(len(panel), "rows | positive rate:", round(panel.target.mean(), 3))
print(panel.groupby("ingredient").target.agg(["size","mean"]).round(3))

# time-based split with a K-month gap
train = panel[panel.m <= "2017-06-01"]
test  = panel[panel.m >= "2018-01-01"]
feats = ["IC025","IC","PRR","ROR_lo","a","new_cases","new_3m","ic025_d3","ic025_d6","age_m","other_cls_sig"]

def p_at_k(df, score, k=50):
    return df.assign(s=score).nlargest(k, "s").target.mean()

res = {}
for name, col in [("baseline: current IC025", "IC025"), ("baseline: linear trend", "trend_K"), ("baseline: new cases 3m", "new_3m")]:
    res[name] = (average_precision_score(test.target, test[col].fillna(-9)), p_at_k(test, test[col].fillna(-9)))

mdl = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=0)
mdl.fit(train[feats], train.target)
p = mdl.predict_proba(test[feats])[:, 1]
res["gradient boosting"] = (average_precision_score(test.target, p), p_at_k(test, p))

print(pd.DataFrame(res, index=["PR-AUC", "precision@50"]).T.round(3))
print("test positive rate (PR-AUC reference):", round(test.target.mean(), 3))

131376 rows | positive rate: 0.038
                size   mean
ingredient                 
canagliflozin  35037  0.032
dapagliflozin  20739  0.041
dulaglutide    27751  0.032
empagliflozin  17729  0.055
ertugliflozin    374  0.217
liraglutide    27499  0.029
semaglutide     2247  0.136
                         PR-AUC  precision@50
baseline: current IC025   0.152          0.34
baseline: linear trend    0.197          0.56
baseline: new cases 3m    0.040          0.00
gradient boosting         0.425          0.68
test positive rate (PR-AUC reference): 0.035


## Closing DB connection

In [9]:
con.close()

# Explicitly close any ambient DuckDB connections
try:
    duckdb.close()
    print("✓ Closed ambient DuckDB connection.")
except Exception as e:
    print(f"Notice: {e}")

✓ Closed ambient DuckDB connection.
